In [ ]:
#Name : Shaza Adelina Binti Khairullah
#ID:SN0107857

In [2]:
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

df = pd.read_csv('npr.csv')
documents = df['Article'].tolist()[:300]

#Preprocess 
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

#Create document-term matrix 
dictionary = corpora.Dictionary(preprocessed_documents)

dictionary.filter_extremes(no_below=15, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

#LDA 
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)




article_labels = []
for i, doc in enumerate(preprocessed_documents):
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability [cite: 227]
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)


df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})
print("Table with Articles and Topic:")
print(df_result.head(10)) # Showing first 10 for brevity
print("\n" + "="*30 + "\n")


print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Table with Articles and Topic:
                                             Article  Topic
0  In the Washington of 2016, even when the polic...      2
1    Donald Trump has used Twitter  —   his prefe...      2
2    Donald Trump is unabashedly praising Russian...      2
3  Updated at 2:50 p. m. ET, Russian President Vl...      2
4  From photography, illustration and video, to d...      3
5  I did not want to join yoga class. I hated tho...      3
6  With a   who has publicly supported the debunk...      3
7  I was standing by the airport exit, debating w...      3
8  If movies were trying to be more realistic, pe...      3
9  Eighteen years ago, on New Year’s Eve, David F...      3


Top Terms for Each Topic:
Topic 0:
- "police" (weight: 0.017)
- "report" (weight: 0.012)
- "officer" (weight: 0.011)
- "reported" (weight: 0.009)
- "city" (weight: 0.009)
- "according" (weight: 0.009)
- "department" (weight: 0.008)
- "told" (weight: 0.008)
- "chicago" (weight: 0.007)
- "news" (weight: 0.00